# Follow-up Adequacy
**We examine censoring rates by diagnosis year to assess whether patients have sufficient follow-up to produce reliable pseudo-observations at 2 years.**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
dtype_map = pd.read_csv('../outputs/pembrochemo_pembro_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
df = pd.read_csv('../outputs/pembrochemo_pembro_features_df.csv', dtype = dtype_map)

In [3]:
df.shape

(1736, 162)

## Censoring rates by years

In [4]:
df.groupby('adv_diagnosis_year')['event'].apply(lambda x: (x == 0).mean())

adv_diagnosis_year
2011    0.500000
2012    0.500000
2013    0.250000
2014    0.272727
2015    0.304348
2016    0.256410
2017    0.258427
2018    0.306122
2019    0.312757
2020    0.344720
2021    0.395210
2022    0.538462
2023    0.837662
Name: event, dtype: float64

In [5]:
(df.query('adv_diagnosis_year >= 2022')['event'] == 0).mean()

np.float64(0.6346555323590815)

In [6]:
results = []
for year in sorted(df['adv_diagnosis_year'].unique()):
    censored = df.query('adv_diagnosis_year == @year and event == 0', engine='python')
    if len(censored) > 0:
        frac = (censored['duration'] < 730).mean()
        results.append({'year': year, 'frac_censored_before_2y': frac})

pd.DataFrame(results)

,year,frac_censored_before_2y
0,2011,0.000000
1,2012,1.000000
2,2013,1.000000
3,2014,1.000000
4,2015,0.428571
5,2016,0.550000
6,2017,0.391304
7,2018,0.400000
8,2019,0.486842
9,2020,0.414414


**Among patients diagnosed with advanced disease in 2022 and later, >60% are censored and they all lack 2-year follow-up from the data cut, making 2-year RMST pseudo-observations entirely extrapolated for these patients. The primary analysis is therefore restricted to patients diagnosed in 2021 or earlier, where some patients have reached the 2-year endpoint and pseudo-observations are grounded in observed rather than extrapolated survival.**